# HMS — Ensemble Submission (PyTorch v12 + Keras Baseline)

Ensembles two model families using different input data:
- **PyTorch v12**: EEG-derived spectrograms (trainable STFT + EfficientNetV2-B2 + GRU)
- **Keras Baseline**: Kaggle-provided spectrograms (EfficientNetV2-B2 image classifier)

In [1]:
!pip install /kaggle/input/datasets/parapapapam/nnaudio031/*.whl -q --no-deps

import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import random, gc
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import timm
from nnAudio.features import STFT
from scipy.signal import butter, sosfiltfilt
import joblib

# Keras/TF
import keras_cv
import keras
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if len(gpus) > 0:
    keras.mixed_precision.set_global_policy("mixed_float16")
    tf.config.optimizer.set_jit(True)

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"TensorFlow: {tf.__version__}, Keras: {keras.__version__}")

2026-03-05 17:19:48.160657: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772731188.579445      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772731188.661005      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772731189.371103      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772731189.371166      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772731189.371169      17 computation_placer.cc:177] computation placer alr

PyTorch: 2.9.0+cpu, CUDA: False
TensorFlow: 2.19.0, Keras: 3.10.0


2026-03-05 17:20:21.933374: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Ensemble Configuration

In [2]:
# ==========================================================================
# ENSEMBLE WEIGHTS — adjust based on individual LB scores
# ==========================================================================
PYTORCH_WEIGHT = 0.9
KERAS_WEIGHT = 0.1

# Shared
NUM_CLASSES = 6
CLASS_NAMES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]

# Competition data
BASE_PATH = Path("/kaggle/input/competitions/hms-harmful-brain-activity-classification")

print(f"Ensemble: {PYTORCH_WEIGHT:.0%} PyTorch + {KERAS_WEIGHT:.0%} Keras")

Ensemble: 90% PyTorch + 10% Keras


## Model Paths

In [3]:
# ============================================================
# ADJUST THESE: point to your uploaded model checkpoints
# ============================================================
PYTORCH_MODEL_DIR = Path("/kaggle/input/models/andrewpearlee/v12/pytorch/default/1")
KERAS_MODEL_PATH = Path("/kaggle/input/models/andrewpearlee/baseline/keras/default/1/best_model.keras")

# Working directories
PROCESSED_EEG_DIR = Path("/kaggle/temp/processed_eeg")
PROCESSED_EEG_DIR.mkdir(parents=True, exist_ok=True)
SPEC_DIR = Path("/kaggle/temp/spectrograms")
SPEC_DIR.mkdir(parents=True, exist_ok=True)

# Find PyTorch checkpoints
pytorch_ckpts = sorted(PYTORCH_MODEL_DIR.glob("best_model_*.pt"))
print(f"PyTorch checkpoints: {len(pytorch_ckpts)}")
for f in pytorch_ckpts:
    print(f"  {f.name}")
print(f"Keras model: {KERAS_MODEL_PATH.name} (exists={KERAS_MODEL_PATH.exists()})")

PyTorch checkpoints: 5
  best_model_v12_fold0.pt
  best_model_v12_fold1.pt
  best_model_v12_fold2.pt
  best_model_v12_fold3.pt
  best_model_v12_fold4.pt
Keras model: best_model.keras (exists=True)


## Load Test Metadata

In [4]:
test_df = pd.read_csv(BASE_PATH / "test.csv")
test_df["eeg_path"] = test_df["eeg_id"].apply(
    lambda x: str(BASE_PATH / "test_eegs" / f"{x}.parquet"))
test_df["spec2_path"] = test_df["spectrogram_id"].apply(
    lambda x: str(SPEC_DIR / f"{x}.npy"))
print(f"Test samples: {len(test_df)}")

Test samples: 1


---
# Part A: PyTorch v12 (EEG → Trainable STFT → CNN → GRU)

In [5]:
BIPOLAR_MONTAGE = {
    "LL": [("Fp1","F7"), ("F7","T3"), ("T3","T5"), ("T5","O1")],
    "RL": [("Fp2","F8"), ("F8","T4"), ("T4","T6"), ("T6","O2")],
    "LP": [("Fp1","F3"), ("F3","C3"), ("C3","P3"), ("P3","O1")],
    "RP": [("Fp2","F4"), ("F4","C4"), ("C4","P4"), ("P4","O2")],
}
BIPOLAR_PAIRS = []
for chain in ["LL", "RL", "LP", "RP"]:
    BIPOLAR_PAIRS.extend(BIPOLAR_MONTAGE[chain])
print(f"Bipolar channels: {len(BIPOLAR_PAIRS)}")

Bipolar channels: 16


In [6]:
class PytorchCFG:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    num_classes = 6
    eeg_sample_rate = 200
    eeg_duration = 50
    eeg_samples = eeg_sample_rate * eeg_duration
    bandpass_low = 0.5
    bandpass_high = 20.0
    bandpass_order = 4
    n_fft = 512
    hop_length = 128
    zoom_duration = 10
    zoom_hop_length = 32
    freq_crop_hz = 20.0
    trainable_stft = True
    backbone = "tf_efficientnetv2_b2"
    gru_hidden = 256
    gru_layers = 2
    dropout = 0.35
    spec_freq_mask = 10
    spec_time_mask = 10
    spec_num_masks = 2
    batch_size = 16
    num_workers = 0
    use_amp = False

In [7]:
def bandpass_filter(data, low, high, fs, order=4):
    nyq = fs / 2.0
    sos = butter(order, [low / nyq, high / nyq], btype='band', output='sos')
    filtered = np.zeros_like(data)
    for i in range(data.shape[0]):
        if np.std(data[i]) > 1e-6:
            try:
                filtered[i] = sosfiltfilt(sos, data[i]).astype(np.float32)
            except ValueError:
                filtered[i] = data[i]
    return filtered


def preprocess_eeg(row_idx, dataframe):
    row = dataframe.iloc[row_idx]
    eeg_id = row["eeg_id"]
    output_path = PROCESSED_EEG_DIR / f"{eeg_id}.npz"
    if output_path.exists():
        return
    eeg_df = pd.read_parquet(row["eeg_path"])
    offset = int(row.get("eeg_label_offset_seconds", 0))
    start = offset * PytorchCFG.eeg_sample_rate
    end = start + PytorchCFG.eeg_samples
    window = eeg_df.iloc[start:end]
    if len(window) < PytorchCFG.eeg_samples:
        pad = pd.DataFrame(np.zeros((PytorchCFG.eeg_samples - len(window), len(window.columns))),
                           columns=window.columns)
        window = pd.concat([window, pad], ignore_index=True)
    cols = window.columns.tolist()
    bipolar = []
    for (a, b) in BIPOLAR_PAIRS:
        sig = window[a].values - window[b].values if (a in cols and b in cols) else np.zeros(PytorchCFG.eeg_samples, dtype=np.float32)
        bipolar.append(sig)
    bipolar = np.stack(bipolar, axis=0).astype(np.float32)
    bipolar = np.nan_to_num(bipolar, nan=0.0)
    bipolar = np.clip(bipolar, -1024, 1024)
    bipolar = bandpass_filter(bipolar, PytorchCFG.bandpass_low, PytorchCFG.bandpass_high,
                              PytorchCFG.eeg_sample_rate, PytorchCFG.bandpass_order)
    chan_mean = bipolar.mean(axis=1).astype(np.float32)
    chan_std = bipolar.std(axis=1).astype(np.float32)
    stats = np.concatenate([chan_mean, chan_std])
    bipolar = (bipolar - chan_mean[:, None]) / (chan_std[:, None] + 1e-6)
    np.savez_compressed(str(output_path), eeg=bipolar, stats=stats)

unique_test = test_df.drop_duplicates(subset="eeg_id").reset_index(drop=True)
print(f"Processing {len(unique_test)} test EEGs...")
_ = joblib.Parallel(n_jobs=-1, backend="loky")(
    joblib.delayed(preprocess_eeg)(i, unique_test) for i in tqdm(range(len(unique_test))))
print("Done.")

Processing 1 test EEGs...


  0%|          | 0/1 [00:00<?, ?it/s]

Done.


In [8]:
class HMSDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.npz_paths = [str(PROCESSED_EEG_DIR / f"{eid}.npz") for eid in self.df["eeg_id"].values]
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        data = np.load(self.npz_paths[idx])
        return torch.tensor(data["eeg"], dtype=torch.float32), torch.tensor(data["stats"], dtype=torch.float32)

pytorch_test_ds = HMSDataset(test_df)
pytorch_test_loader = DataLoader(pytorch_test_ds, batch_size=min(PytorchCFG.batch_size, len(test_df)),
                                  shuffle=False, num_workers=PytorchCFG.num_workers, pin_memory=True)
print(f"PyTorch test batches: {len(pytorch_test_loader)}")

PyTorch test batches: 1


In [9]:
class SpectrogramModel(nn.Module):
    def __init__(self, cfg=PytorchCFG):
        super().__init__()
        self.cfg = cfg
        freq_res = cfg.eeg_sample_rate / cfg.n_fft
        self.max_bin = int(cfg.freq_crop_hz / freq_res) + 1
        self.zoom_samples = cfg.zoom_duration * cfg.eeg_sample_rate
        self.zoom_start = (cfg.eeg_samples - self.zoom_samples) // 2
        self.zoom_end = self.zoom_start + self.zoom_samples

        self.stft_wide = STFT(n_fft=cfg.n_fft, hop_length=cfg.hop_length,
                              sr=cfg.eeg_sample_rate, trainable=cfg.trainable_stft,
                              output_format="Magnitude")
        self.stft_zoom = STFT(n_fft=cfg.n_fft, hop_length=cfg.zoom_hop_length,
                              sr=cfg.eeg_sample_rate, trainable=cfg.trainable_stft,
                              output_format="Magnitude")
        self.log_eps = 1e-6
        self.backbone = timm.create_model(cfg.backbone, pretrained=False, in_chans=3, features_only=True)
        img_h = 16 * self.max_bin
        with torch.no_grad():
            backbone_out = self.backbone(torch.randn(1, 3, img_h, 64))
        backbone_ch = backbone_out[-1].shape[1]
        self.gru = nn.GRU(input_size=backbone_ch, hidden_size=cfg.gru_hidden,
                          num_layers=cfg.gru_layers, batch_first=True, bidirectional=True)
        gru_out = cfg.gru_hidden * 2
        self.time_attn = nn.Linear(gru_out, 1)
        self.stats_mlp = nn.Sequential(nn.Linear(32, 64), nn.ReLU(),
                                        nn.Dropout(cfg.dropout), nn.Linear(64, 32))
        self.head = nn.Sequential(nn.Dropout(cfg.dropout), nn.Linear(gru_out + 32, 128), nn.ReLU(),
                                   nn.Dropout(cfg.dropout), nn.Linear(128, cfg.num_classes))

    def _run_stft(self, eeg, stft_layer):
        B, C, T = eeg.shape
        x = stft_layer(eeg.reshape(B * C, T))[:, :self.max_bin, :]
        return x.reshape(B, C, x.shape[1], x.shape[2])

    def _log_norm(self, spec):
        x = torch.log(spec.clamp(min=self.log_eps))
        return (x - x.mean(dim=(1, 2), keepdim=True)) / (x.std(dim=(1, 2), keepdim=True) + 1e-6)

    def spec_augment(self, spec):
        if not self.training:
            return spec
        B, H, T = spec.shape
        for _ in range(self.cfg.spec_num_masks):
            f = random.randint(0, self.cfg.spec_freq_mask)
            f0 = random.randint(0, max(H - f, 1))
            spec[:, f0:f0+f, :] = 0
            t = random.randint(0, self.cfg.spec_time_mask)
            t0 = random.randint(0, max(T - t, 1))
            spec[:, :, t0:t0+t] = 0
        return spec

    def make_multiscale_image(self, eeg):
        wide = self._log_norm(self._run_stft(eeg, self.stft_wide).flatten(1, 2))
        zoom = self._run_stft(eeg[:, :, self.zoom_start:self.zoom_end], self.stft_zoom).flatten(1, 2)
        zoom = self._log_norm(zoom)
        zoom = F.interpolate(zoom.unsqueeze(1), size=(wide.shape[1], wide.shape[2]),
                             mode='bilinear', align_corners=False).squeeze(1)
        wide = self.spec_augment(wide)
        zoom = self.spec_augment(zoom)
        return torch.stack([wide, zoom, (wide + zoom) / 2], dim=1)

    def forward(self, eeg, stats):
        img = self.make_multiscale_image(eeg)
        fmap = self.backbone(img)[-1]
        x = fmap.mean(dim=2).permute(0, 2, 1).float()
        x, _ = self.gru(x)
        w = torch.softmax(self.time_attn(x), dim=1)
        x = (x * w).sum(dim=1)
        s = self.stats_mlp(stats.float())
        logits = self.head(torch.cat([x, s], dim=1))
        return F.softmax(logits, dim=1)

print("SpectrogramModel defined.")

SpectrogramModel defined.


### PyTorch Inference

In [10]:
@torch.no_grad()
def pytorch_predict(model, loader, device):
    model.eval()
    preds = []
    for eeg, stats in tqdm(loader, desc="  PyTorch predicting"):
        with autocast(enabled=PytorchCFG.use_amp):
            p = model(eeg.to(device), stats.to(device))
        preds.append(p.cpu().numpy())
    return np.concatenate(preds, axis=0)

pytorch_preds_list = []
for ckpt_path in pytorch_ckpts:
    print(f"\n{ckpt_path.stem}:")
    gc.collect(); torch.cuda.empty_cache()
    model = SpectrogramModel(PytorchCFG).to(PytorchCFG.device)
    ckpt = torch.load(str(ckpt_path), map_location=PytorchCFG.device)
    model.load_state_dict(ckpt["model_state_dict"])
    info = []
    if ckpt.get("val_loss"): info.append(f"val_loss={ckpt['val_loss']:.4f}")
    if ckpt.get("val_loss_hq"): info.append(f"val_hq={ckpt['val_loss_hq']:.4f}")
    print(f"  Loaded ({', '.join(info) if info else 'ok'})")
    pytorch_preds_list.append(pytorch_predict(model, pytorch_test_loader, PytorchCFG.device))
    del model; gc.collect(); torch.cuda.empty_cache()

pytorch_preds = np.mean(pytorch_preds_list, axis=0)
pytorch_preds = pytorch_preds / pytorch_preds.sum(axis=1, keepdims=True)
print(f"\nPyTorch ensemble: {len(pytorch_preds_list)} model(s), sum={pytorch_preds[0].sum():.4f}")


best_model_v12_fold0:
STFT kernels created, time used = 0.1065 seconds
STFT kernels created, time used = 0.0107 seconds
  Loaded (val_loss=0.6655, val_hq=0.6970)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  PyTorch predicting:   0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_17/1226257201.py:6: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=PytorchCFG.use_amp):



best_model_v12_fold1:
STFT kernels created, time used = 0.0109 seconds
STFT kernels created, time used = 0.0092 seconds
  Loaded (val_loss=0.6796, val_hq=0.6988)


...


best_model_v12_fold3:
STFT kernels created, time used = 0.0100 seconds
STFT kernels created, time used = 0.0097 seconds
  Loaded (val_loss=0.7053, val_hq=0.7380)


  PyTorch predicting:   0%|          | 0/1 [00:00<?, ?it/s]


best_model_v12_fold4:
STFT kernels created, time used = 0.0096 seconds
STFT kernels created, time used = 0.0087 seconds
  Loaded (val_loss=0.6876, val_hq=0.7244)


  PyTorch predicting:   0%|          | 0/1 [00:00<?, ?it/s]


PyTorch ensemble: 5 model(s), sum=1.0000


---
# Part B: Keras Baseline (Kaggle Spectrograms → EfficientNetV2-B2)

In [11]:
def process_spec(spec_id):
    output_path = SPEC_DIR / f"{spec_id}.npy"
    if output_path.exists():
        return
    spec_path = BASE_PATH / "test_spectrograms" / f"{spec_id}.parquet"
    spec = pd.read_parquet(str(spec_path))
    spec = spec.fillna(0).values[:, 1:].T
    spec = spec.astype("float32")
    np.save(str(output_path), spec)

spec_ids = test_df["spectrogram_id"].unique()
print(f"Processing {len(spec_ids)} test spectrograms...")
_ = joblib.Parallel(n_jobs=-1, backend="loky")(
    joblib.delayed(process_spec)(sid) for sid in tqdm(spec_ids, total=len(spec_ids)))
print("Done.")

Processing 1 test spectrograms...


  0%|          | 0/1 [00:00<?, ?it/s]

Done.


In [12]:
class KerasCFG:
    preset = "efficientnetv2_b2_imagenet"
    image_size = [400, 300]
    batch_size = 32
    num_classes = 6

def build_decoder(dtype=32):
    def decode_signal(path):
        file_bytes = tf.io.read_file(path)
        sig = tf.io.decode_raw(file_bytes, tf.float32)
        sig = sig[1024 // dtype:]
        sig = tf.reshape(sig, [400, -1])
        sig = tf.clip_by_value(sig, tf.math.exp(-4.0), tf.math.exp(8.0))
        sig = tf.math.log(sig)
        sig -= tf.math.reduce_mean(sig)
        sig /= tf.math.reduce_std(sig) + 1e-6
        sig = tf.tile(sig[..., None], [1, 1, 3])
        return sig
    return decode_signal

AUTO = tf.data.experimental.AUTOTUNE
test_paths = test_df.spec2_path.values
keras_test_ds = tf.data.Dataset.from_tensor_slices(test_paths)
keras_test_ds = keras_test_ds.map(build_decoder(), num_parallel_calls=AUTO)
keras_test_ds = keras_test_ds.batch(min(KerasCFG.batch_size, len(test_df)), drop_remainder=False)
keras_test_ds = keras_test_ds.prefetch(AUTO)

# Build and load model
keras_model = keras_cv.models.ImageClassifier.from_preset(
    KerasCFG.preset, num_classes=KerasCFG.num_classes)
keras_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                     loss=keras.losses.KLDivergence())
keras_model.load_weights(str(KERAS_MODEL_PATH))
print(f"Keras model loaded: {KERAS_MODEL_PATH.name}")

# Predict
keras_preds = keras_model.predict(keras_test_ds)
keras_preds = keras_preds.astype(np.float64)
keras_preds = np.exp(keras_preds - keras_preds.max(axis=1, keepdims=True))
keras_preds = keras_preds / keras_preds.sum(axis=1, keepdims=True)
keras_preds = np.clip(keras_preds, 1e-15, 1.0)
keras_preds = keras_preds / keras_preds.sum(axis=1, keepdims=True)
print(f"Keras predictions: {keras_preds.shape}, sum={keras_preds[0].sum():.4f}")

del keras_model; gc.collect()
tf.keras.backend.clear_session()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 626 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Keras model loaded: best_model.keras
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Keras predictions: (1, 6), sum=1.0000


---
# Ensemble

In [13]:
# Weighted ensemble
preds = PYTORCH_WEIGHT * pytorch_preds + KERAS_WEIGHT * keras_preds
preds = preds / preds.sum(axis=1, keepdims=True)

print(f"Ensemble predictions: {preds.shape}")
print(f"Prob sum: {preds[0].sum():.4f}")
print(f"\nPer-model predictions for sample 0:")
print(f"  PyTorch: {np.round(pytorch_preds[0], 4)}")
print(f"  Keras:   {np.round(keras_preds[0], 4)}")
print(f"  Ensemble:{np.round(preds[0], 4)}")

Ensemble predictions: (1, 6)
Prob sum: 1.0000

Per-model predictions for sample 0:
  PyTorch: [0.0742 0.0189 0.0064 0.111  0.154  0.6355]
  Keras:   [0.1572 0.1954 0.1408 0.1927 0.1486 0.1653]
  Ensemble:[0.0825 0.0365 0.0198 0.1192 0.1535 0.5885]


## Submission

In [14]:
target_cols = [x.lower() + "_vote" for x in CLASS_NAMES]
pred_df = test_df[["eeg_id"]].copy()
pred_df[target_cols] = preds.tolist()

sub_df = pd.read_csv(BASE_PATH / "sample_submission.csv")[["eeg_id"]]
sub_df = sub_df.merge(pred_df, on="eeg_id", how="left")

sub_df.to_csv("submission.csv", index=False)
print(f"submission.csv saved ({len(sub_df)} rows)")
print(sub_df.head())

submission.csv saved (1 rows)
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  3911565283      0.082512  0.036534  0.019831   0.119181   0.153463   

   other_vote  
0    0.588478  
